## CLASSIFICATION ON SUBSTANCES FROM MOLECULAR DESCRIPTORS & BEHAVIOURAL CLASSES

#### 0. Initialisation:

In [1]:
#install.packages("fda")
#install.packages("hash")

In [2]:
library("caret")
library("e1071")
library("fda")
library("ggplot2")
library("nnet")
library("pls")
library("psychTools")
library("randomForest")
library("readr")
library("splines")
library("stringr")
library("tibble")
library("tidymodels")
library("VGAM")
library("xgboost")

Le chargement a nécessité le package : ggplot2

Le chargement a nécessité le package : lattice


Attachement du package : ‘e1071’


L'objet suivant est masqué depuis ‘package:ggplot2’:

    element


Le chargement a nécessité le package : splines

Le chargement a nécessité le package : fds

Le chargement a nécessité le package : rainbow

Le chargement a nécessité le package : MASS

Le chargement a nécessité le package : pcaPP

Le chargement a nécessité le package : RCurl

Le chargement a nécessité le package : deSolve


Attachement du package : ‘fda’


L'objet suivant est masqué depuis ‘package:lattice’:

    melanoma


L'objet suivant est masqué depuis ‘package:graphics’:

    matplot


L'objet suivant est masqué depuis ‘package:datasets’:

    gait



Attachement du package : ‘pls’


L'objet suivant est masqué depuis ‘package:caret’:

    R2


L'objet suivant est masqué depuis ‘package:stats’:

    loadings


randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


#### 1. Import data:

In [3]:
## import m.d. database (gLASSO done on it to select + imp m.d. & to reduce dimensionality of the df):
md_df <- read.csv("../../pipeline_1/data/glasso_results_nomet.csv", row.names = 'name')
md_data <- subset(md_df, select = -X)
head(md_data) ## 89 m.d. and 33 sub

## import beh. classes database:
beh_classes <- read.csv("../data/classes_df_nomet_behaviour_K7_IGT.csv", row.names = 'substance')
head(beh_classes)

,X3,X5,X6,X15,X16,X17,X18,X20,X21,X22,⋯,X151,X157,X161,X162,X168,X172,X173,X177,X190,X205
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Carbaryl,-0.08962149,-0.3636591,-0.3302214,0.2427118,0.9731771,1.6317622,-0.8484532,-0.03981839,-0.1878516,0.2168568,⋯,-0.2390457,-0.5074461,-0.2261335,-0.2970443,0.03889876,-0.2390457,0.5141872,-0.2970443,-0.1666667,-0.3481553
Cypermethrine,-0.48017021,0.4114824,0.9332837,0.2638783,0.8726667,1.2574848,0.6187160,1.32052350,-0.4518745,0.9106901,⋯,-0.2390457,-0.5074461,-0.2261335,-0.2970443,-0.44085263,4.1833001,1.4201361,-0.2970443,-0.1666667,-0.3481553
DDD(2-4),-0.14550275,-0.2681413,0.3678056,-0.9179209,-1.0758237,-1.0395508,0.6187307,0.35214692,-0.6468554,0.4014187,⋯,-0.2390457,0.7874163,-0.2261335,-0.2970443,-0.44085263,-0.2390457,-0.3917617,-0.2970443,-0.1666667,-0.3481553
Dicofol,-1.11434449,-0.1734469,0.6641444,-0.8314908,-1.0634803,-1.0806480,0.6220778,1.03362163,-1.2319861,0.6936119,⋯,-0.2390457,1.4348476,-0.2261335,-0.2970443,-0.44085263,-0.2390457,-0.3917617,-0.2970443,-0.1666667,-0.3481553
Isodrin,-0.70877150,3.2561384,0.6313875,-0.4240347,-0.4586548,-0.3203512,0.6188102,1.87447792,-1.2901015,1.3216239,⋯,-0.2390457,2.0822788,-0.2261335,-0.2970443,3.87690990,-0.2390457,-0.3917617,-0.2970443,-0.1666667,-0.3481553
Lindane,-0.08962149,1.1371790,1.3783030,-0.5475062,-0.4586548,-0.1491132,0.6160823,0.96183760,-1.3027532,0.5080324,⋯,-0.2390457,3.3771412,-0.2261335,-0.2970443,0.03889876,-0.2390457,0.5141872,-0.2970443,-0.1666667,-0.3481553


,y,substance_type
,<int>,<chr>
1-2Dichloroethane,5,solvent
1-chlorodecane,4,other
124-Trichlorobenzene,4,solvent
4-octylphenol,3,other
A736,4,other
Acetone,4,solvent


In [4]:
dim(md_data)
dim(beh_classes)

[1] 33 89

[1] 33  2

In [5]:
table(beh_classes$substance_type)


    insecticide           other other pesticide             PAH            PPCP 
             10               7               4               2               2 
        solvent 
              8 

In [6]:
#rownames(md_data)[rownames(md_data) == "Chlorothanolil"] <- "Chlorothalonil"
rownames(beh_classes)[rownames(beh_classes) == "Chlorothanolil"] <- "Chlorothalonil"

In [7]:
md_data <- md_data[rownames(md_data) %in% rownames(beh_classes), ]

In [8]:
md_data$y <- list(NULL)
md_data$y <- beh_classes$y[match(rownames(md_data), rownames(beh_classes))]

#### 2. Prepare classification models:

In [9]:
names <- rownames(md_data)
X <- subset(md_data, select = -y)
y <- md_data$y
y <- as.factor(y)

In [10]:
## sanity checks:
colnames(X) ## all good
table(y)

[1] "X3"   "X5"   "X6"   "X15"  "X16"  "X17"  "X18"  "X20"  "X21"  "X22" 
[11] "X23"  "X25"  "X26"  "X28"  "X41"  "X42"  "X44"  "X47"  "X48"  "X49" 
[21] "X50"  "X51"  "X52"  "X53"  "X54"  "X55"  "X58"  "X59"  "X60"  "X61" 
[31] "X62"  "X64"  "X65"  "X66"  "X67"  "X68"  "X70"  "X71"  "X72"  "X73" 
[41] "X74"  "X75"  "X76"  "X77"  "X78"  "X79"  "X81"  "X83"  "X84"  "X85" 
[51] "X87"  "X88"  "X89"  "X93"  "X94"  "X95"  "X97"  "X99"  "X101" "X102"
[61] "X103" "X104" "X109" "X110" "X111" "X113" "X114" "X115" "X117" "X119"
[71] "X120" "X121" "X122" "X124" "X129" "X130" "X142" "X147" "X148" "X151"
[81] "X157" "X161" "X162" "X168" "X172" "X173" "X177" "X190" "X205"

y
 1  2  3  4  5  6 
 3  4  2 21  2  1 

#### 3. Model pipeline:

Here, in train: put all substances and exclude one sub at each iteration to put it in the test set, so that we test this substance against all the other substances used. Allows for better predictions and more reliable results. Then, return a dataframe assembling all the results obtained, as well as the mean accuracy across ALL classification models used.

In [11]:
print(sum(is.na(md_data)))
print(sum(is.na(beh_classes)))

[1] 0
[1] 0


In [12]:
#### MODEL LOOPING:
## create empty matrix:
preds <- matrix(0, nrow = 0, ncol = 5) ## 5 cols = 1 col for y + 4 cols for model predictions

for (sub_test in rownames(md_data)) {
    ## prepare train/test sets:
    test  <- md_data[sub_test, , drop = FALSE]  # drop=FALSE keeps dataframe format
    train <- md_data[rownames(md_data) != sub_test, ]
    
    ## convert y to factor:
    train$y <- factor(train$y)
    test$y  <- factor(test$y)
    all_levels <- union(levels(train$y), levels(test$y))
    train$y <- factor(train$y, levels = all_levels)
    test$y  <- factor(test$y,  levels = all_levels)

    train$y <- droplevels(train$y)
    test$y <- droplevels(test$y)
    
    ## a. Logistic Regression:
    set.seed(120)
    model_LR <- multinom(y ~ ., data = train, trace = FALSE, MaxNWts = 2000)
    y_pred_LR_test <- model_LR %>% predict(test)
    
    ## b. Random Forest:
    set.seed(120)
    y_pred_RF <- tryCatch({
        model_RF <- randomForest(y  ~ ., data = train, ntree = 20, mtry = 2)
        predict(model_RF, newdata = test)
    }, error = function(e){
        message("Model failed for ", sub_test, ":", e$message)
        NA
    })
    
    ## c. SVM:
    ## c1. linear:
    set.seed(120)
    model_SVM_linear <- svm(y ~ ., data = train, kernel = "linear", cost = 15)
    y_pred_SVM_l <- predict(model_SVM_linear, test)
    
    ## c2. non-linear => RBF kernel (gaussian-like), best for our p > n case (!!):
    set.seed(120)
    model_SVM_nonlinear <- svm(y ~ ., data = train, kernel = "radial", cost = 15)  
    y_pred_SVM_nl <- predict(model_SVM_nonlinear, test)
    
    ## bind results together to make final df with pred class for all sub tested separately:
    ys <- cbind(as.character(test$y), as.character(y_pred_LR_test), as.character(y_pred_RF), as.character(y_pred_SVM_l), as.character(y_pred_SVM_nl))#, as.character(y_pred_fLDA))
    preds <- rbind(preds, ys)
}

Warning message in svm.default(x, y, scale = scale, ..., na.action = na.action):
“Variable(s) ‘X190’ constant. Cannot scale data.”
Warning message in svm.default(x, y, scale = scale, ..., na.action = na.action):
“Variable(s) ‘X190’ constant. Cannot scale data.”


In [13]:
## make final df to output results:
preds_df <- as.data.frame(preds, row.names = rownames(md_data))
colnames(preds_df) <- c("true_y", "fLR", "fRF", "fSVM_l", "fSVM_nl")
View(preds_df)

,true_y,fLR,fRF,fSVM_l,fSVM_nl
,<chr>,<chr>,<chr>,<chr>,<chr>
Carbaryl,4,6,4,6,4
Cypermethrine,2,3,4,4,4
DDD(2-4),3,4,4,4,4
Dicofol,4,3,4,3,3
Isodrin,2,6,4,4,4
Lindane,2,4,4,4,4
Methomyl,6,4,4,4,4
Tebufenozide,4,3,4,3,4
alphaEndosulfan,4,4,4,4,4


In [14]:
table(predicted = preds_df$fRF, actual = preds_df$true_y)

         actual
predicted  1  2  3  4  5  6
        4  3  4  2 21  2  1

In [15]:
## compute some accuracy metrics:
# overall accuracy (includes singletons)
mean(preds_df$true_y == preds_df$fLR, na.rm = TRUE)
mean(preds_df$true_y == preds_df$fRF, na.rm = TRUE)
mean(preds_df$true_y == preds_df$fSVM_l, na.rm = TRUE)
mean(preds_df$true_y == preds_df$fSVM_nl, na.rm = TRUE)

[1] 0.2424242

[1] 0.6363636

[1] 0.3939394

[1] 0.5454545

In [16]:
## convert df to LaTeX table:
df2latex(preds_df)

% df2latex % preds_df 
\begin{table}[htpb]\caption{df2latex}
\begin{center}
\begin{scriptsize} 
\begin{tabular} {l r r r r r }
 \multicolumn{ 5 }{l}{ A table from the psych package in R } \cr 
 \hline Variable  &   {tru_y} &  {fLR} &  {fRF} &  {fSVM_l} &  {fSVM_n}\cr 
  \hline 
Carbaryl   &  4  &  6  &  4  &  6  &  4 \cr 
 Cypermethrine   &  2  &  3  &  4  &  4  &  4 \cr 
 DDD(2-4)   &  3  &  4  &  4  &  4  &  4 \cr 
 Dicofol   &  4  &  3  &  4  &  3  &  3 \cr 
 Isodrin   &  2  &  6  &  4  &  4  &  4 \cr 
 Lindane   &  2  &  4  &  4  &  4  &  4 \cr 
 Methomyl   &  6  &  4  &  4  &  4  &  4 \cr 
 Tebufenozide   &  4  &  3  &  4  &  3  &  4 \cr 
 alphaEndosulfan   &  4  &  4  &  4  &  4  &  4 \cr 
 betaEndosulfan   &  4  &  5  &  4  &  4  &  4 \cr 
 1-chlorodecane   &  4  &  3  &  4  &  3  &  3 \cr 
 4-octylphenol   &  3  &  4  &  4  &  4  &  4 \cr 
 A736   &  4  &  5  &  4  &  2  &  4 \cr 
 Biphenyl   &  4  &  5  &  4  &  4  &  4 \cr 
 H40   &  1  &  6  &  4  &  6  &  4 \cr 
 L1000   & 